In [6]:
# Leak-Induced Transient Energy Propagation (MSSP-style) — end-to-end pipeline in ONE CELL
# Author: You (UMAR) + ChatGPT
#
# What this does:
# 1) Recursively loads signals from your dataset folders (Accelerometer, Dynamic Pressure Sensor, Hydrophone).
# 2) Parses labels from filenames (Topology, LeakType, FlowCondition, Noise flag, SensorID).
# 3) Computes time-frequency energy propagation using STFT (and optional CWT).
# 4) Quantifies transient energy propagation metrics:
#       - Energy jump ratio (transient vs steady baseline)
#       - Energy rise time
#       - Time-varying spectral centroid and its slope during transient
#       - Band-energy evolution (low/mid/high bands)
# 5) Produces plots and exports a summary CSV for paper tables/figures.
#
# IMPORTANT:
# - Your Windows paths are E:\... which this script supports.
# - Hydrophone .raw sampling rate is not embedded; set FS_HYDROPHONE below (often 44100 or 48000).
# - CSV sampling rate is inferred from a time column if present; otherwise you must set FS_CSV_DEFAULT.
#
# Install deps if needed:
#   pip install numpy pandas scipy matplotlib pywavelets tqdm

import os, re, glob, math, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Dict, Tuple, List
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy import signal

try:
    import pywt  # optional CWT
    HAS_PYWT = True
except Exception:
    HAS_PYWT = False

# =========================
# ========== CONFIG =======
# =========================
ROOTS = [
    r"E:\Upwork Project\AI_Leak_Detection_Project\data\raw\Accelerometer",
    r"E:\Upwork Project\AI_Leak_Detection_Project\data\raw\Dynamic Pressure Sensor",
    r"E:\Upwork Project\AI_Leak_Detection_Project\data\raw\Hydrophone",
]

OUT_DIR = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy"
os.makedirs(OUT_DIR, exist_ok=True)

# Hydrophone raw data sampling rate (set correctly for your testbed)
FS_HYDROPHONE = 44100  # common: 44100 or 48000. Adjust if you know exact.
# If CSV files do not include a time column, this fallback is used:
FS_CSV_DEFAULT = 10240  # typical NI-9234 configs are often 5120/10240/20480 etc. Adjust if known.

# Analysis windowing
SIGNAL_DURATION_SEC = 30.0  # per dataset description
# STFT parameters (work across modalities)
STFT_NPERSEG = 2048
STFT_NOVERLAP = 1536

# Bands for energy propagation (Hz) — broad and robust across sensors; adjust after you inspect PSDs
BANDS = {
    "low": (0.0, 200.0),
    "mid": (200.0, 1000.0),
    "high": (1000.0, 5000.0),
}

# Transient detection configuration:
# If filename says transient (e.g., includes "TR" or "TRANS" etc.), we treat as transient.
# Otherwise we auto-detect an abrupt change in broadband STFT energy.
AUTO_TRANSIENT_DETECT = True
TRANSIENT_DETECT_SMOOTH_SEC = 0.20
TRANSIENT_DETECT_THRESHOLD_Z = 4.0  # higher = stricter, fewer detections

# Plot control
MAKE_PLOTS = True
MAX_PLOTS = 60  # to avoid generating too many; increase if you want everything
PLOT_DPI = 160

# Optional CWT (slower). Enable only if you want wavelet ridge visuals.
DO_CWT = False and HAS_PYWT
CWT_WAVELET = "morl"

np.seterr(all="ignore")

# =========================
# ======= UTILITIES =======
# =========================

def safe_mkdir(p): 
    os.makedirs(p, exist_ok=True)
    return p

def zscore(x):
    x = np.asarray(x, dtype=float)
    mu, sd = np.nanmean(x), np.nanstd(x)
    return (x - mu) / (sd + 1e-12)

def robust_mad(x):
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-12
    return med, mad

def parse_flow_token(tok: str) -> str:
    tok = tok.strip().upper()
    tok = tok.replace(" ", "")
    # normalize a few patterns
    if "LPS" in tok:
        tok = tok.replace("LPS", " LPS")
    return tok

def infer_fs_from_csv(df: pd.DataFrame) -> Optional[float]:
    # Try to infer from a time column
    # Common patterns: "time", "Time", first column increasing
    cols = list(df.columns)
    if len(cols) < 1:
        return None

    # If an explicit time column exists
    for c in cols[:3]:
        if str(c).lower() in ["time", "t", "timestamp", "seconds", "sec"]:
            t = pd.to_numeric(df[c], errors="coerce").to_numpy()
            t = t[np.isfinite(t)]
            if len(t) > 10:
                dt = np.diff(t)
                dt = dt[np.isfinite(dt)]
                if len(dt) > 10 and np.nanmedian(dt) > 0:
                    return float(1.0 / np.nanmedian(dt))

    # Otherwise, if first column looks like time
    if len(cols) >= 2:
        first = pd.to_numeric(df[cols[0]], errors="coerce").to_numpy()
        second = pd.to_numeric(df[cols[1]], errors="coerce").to_numpy()
        # If first is monotonic increasing and second is signal-like
        f = first[np.isfinite(first)]
        if len(f) > 100 and np.all(np.diff(f[:200]) > 0):
            dt = np.diff(f)
            dt = dt[np.isfinite(dt)]
            if len(dt) > 10 and np.nanmedian(dt) > 0:
                return float(1.0 / np.nanmedian(dt))
    return None

def load_csv_signal(path: str) -> Tuple[np.ndarray, float]:
    df = pd.read_csv(path)
    if df.shape[1] == 1:
        x = pd.to_numeric(df.iloc[:, 0], errors="coerce").to_numpy()
        fs = FS_CSV_DEFAULT
        x = x[np.isfinite(x)]
        return x.astype(float), float(fs)

    # Prefer last numeric column as signal if first is time
    numeric_cols = []
    for c in df.columns:
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().mean() > 0.8:
            numeric_cols.append(c)

    if len(numeric_cols) == 0:
        # fallback: try all values
        x = df.to_numpy().reshape(-1)
        x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy()
        x = x[np.isfinite(x)]
        return x.astype(float), float(FS_CSV_DEFAULT)

    fs = infer_fs_from_csv(df) or FS_CSV_DEFAULT

    # Choose a column that is NOT time-like if possible
    time_like = set(["time", "t", "timestamp", "seconds", "sec"])
    candidate_cols = [c for c in numeric_cols if str(c).lower() not in time_like]
    if len(candidate_cols) == 0:
        candidate_cols = numeric_cols

    # If first numeric col looks like time (monotonic), pick the next
    chosen = candidate_cols[-1]
    if len(candidate_cols) >= 2:
        c0 = candidate_cols[0]
        v0 = pd.to_numeric(df[c0], errors="coerce").to_numpy()
        v0 = v0[np.isfinite(v0)]
        if len(v0) > 200 and np.all(np.diff(v0[:200]) > 0):
            chosen = candidate_cols[1]

    x = pd.to_numeric(df[chosen], errors="coerce").to_numpy()
    x = x[np.isfinite(x)]
    return x.astype(float), float(fs)

def load_raw_hydrophone(path: str, fs: float) -> Tuple[np.ndarray, float]:
    # We don't know the encoding; common is 16-bit PCM little-endian.
    # If your raw is float32, change dtype to np.float32.
    # If signal looks clipped/steppy, adjust dtype.
    x = np.fromfile(path, dtype=np.int16).astype(np.float32)
    # normalize to [-1, 1]
    if np.max(np.abs(x)) > 0:
        x = x / (np.max(np.abs(x)) + 1e-12)
    return x.astype(float), float(fs)

def trim_or_pad(x: np.ndarray, fs: float, dur_sec: float) -> np.ndarray:
    n = int(round(fs * dur_sec))
    if len(x) >= n:
        return x[:n]
    # pad with zeros if shorter
    y = np.zeros(n, dtype=float)
    y[:len(x)] = x
    return y

@dataclass
class Meta:
    sensor_group: str      # Accelerometer / Dynamic Pressure Sensor / Hydrophone
    topology: str          # BR / LO
    leak: str              # CC / LC / OL / GL / NL etc.
    flow: str              # 0 LPS / 0.18 LPS / 0.47 LPS / TR ...
    noise: str             # N / ND / unknown
    sensor_id: str         # A1/A2, P1/P2, H1/H2, etc.
    path: str

def parse_meta_from_filename(sensor_group: str, path: str) -> Meta:
    name = os.path.splitext(os.path.basename(path))[0]
    up = name.upper().replace("__", "_").replace("  ", " ")

    # Typical patterns in your description:
    # CSV:  T_L_F_S#  e.g., BR_CC_0.18 LPS_A1
    # Hydrophone: T_L_F_B_S# e.g., LO_OL_ND_N_H2
    toks = re.split(r"[_\s]+", up)

    topology = toks[0] if len(toks) > 0 else "UNK"
    leak = toks[1] if len(toks) > 1 else "UNK"

    # Detect sensor id last token like A1 / P1 / H2
    sensor_id = "UNK"
    if len(toks) >= 1 and re.match(r"^[APH]\d+$", toks[-1]):
        sensor_id = toks[-1]

    # Noise token: often ND or N; in hydrophone includes a background noise marker
    noise = "UNK"
    if "ND" in toks:
        noise = "ND"
    elif "N" in toks:
        # careful: "N" might appear in other contexts; but for hydrophone naming it's common
        noise = "N"

    # Flow token: common: 0, 0.18, 0.47 plus LPS or transient
    # We'll scan for numbers and "LPS", or "TR"/"TRANS"/"TRANSIENT"
    flow = "UNK"
    if any(t in ["TR", "TRANS", "TRANSIENT"] for t in toks):
        flow = "TRANSIENT"
    else:
        # find numeric token possibly followed by LPS
        # reconstruct patterns like "0.18 LPS"
        num = None
        for t in toks:
            if re.match(r"^\d+(\.\d+)?$", t):
                num = t
                break
        if num is not None:
            # check if "LPS" exists anywhere
            if "LPS" in toks:
                flow = f"{num} LPS"
            else:
                # might be 0 without LPS
                flow = num

    return Meta(
        sensor_group=sensor_group,
        topology=topology,
        leak=leak,
        flow=parse_flow_token(flow),
        noise=noise,
        sensor_id=sensor_id,
        path=path
    )

def stft_energy_features(x: np.ndarray, fs: float) -> Dict[str, object]:
    # STFT
    nperseg = min(STFT_NPERSEG, len(x))
    noverlap = min(STFT_NOVERLAP, nperseg - 1) if nperseg > 1 else 0
    f, t, Zxx = signal.stft(
        x, fs=fs, window="hann",
        nperseg=nperseg, noverlap=noverlap, detrend=False, padded=False, boundary=None
    )
    P = (np.abs(Zxx) ** 2)  # power

    # Broadband energy over time
    E_t = np.sum(P, axis=0) + 1e-12  # shape (time,)
    # Spectral centroid over time
    centroid = (np.sum((f[:, None] * P), axis=0) / np.sum(P, axis=0))
    centroid = np.nan_to_num(centroid)

    # Band energies over time
    band_E = {}
    for bn, (lo, hi) in BANDS.items():
        idx = np.where((f >= lo) & (f < hi))[0]
        if len(idx) == 0:
            band_E[bn] = np.zeros_like(E_t)
        else:
            band_E[bn] = np.sum(P[idx, :], axis=0)

    return {
        "f": f, "t": t, "P": P,
        "E_t": E_t, "centroid_t": centroid,
        "band_E_t": band_E
    }

def detect_transient_time(E_t: np.ndarray, t: np.ndarray, fs_time: Optional[float]=None) -> Tuple[bool, float]:
    # Detect abrupt energy change in broadband energy
    # Smooth E_t, compute zscore of derivative, find first exceedance.
    if len(E_t) < 10:
        return (False, float(t[0]) if len(t)>0 else 0.0)

    # smoothing length in samples on STFT time axis
    dt = np.median(np.diff(t)) if len(t) > 2 else 0.05
    win = max(3, int(round(TRANSIENT_DETECT_SMOOTH_SEC / (dt + 1e-12))))
    if win % 2 == 0:
        win += 1

    E_s = signal.savgol_filter(np.log(E_t + 1e-12), window_length=min(win, len(E_t)//2*2+1), polyorder=2, mode="interp")
    dE = np.diff(E_s, prepend=E_s[0])

    med, mad = robust_mad(dE)
    z = (dE - med) / (1.4826 * mad + 1e-12)

    idx = np.where(z > TRANSIENT_DETECT_THRESHOLD_Z)[0]
    if len(idx) == 0:
        return (False, float(t[len(t)//2]))
    return (True, float(t[idx[0]]))

def propagation_metrics(E_t, centroid_t, band_E_t, t, t0):
    # Baseline = before transient start
    # Transient window = from t0 to t0 + 3s (or until end)
    if len(t) < 5:
        return {}

    t = np.asarray(t)
    E_t = np.asarray(E_t)
    centroid_t = np.asarray(centroid_t)

    # define windows
    pre_mask = t < t0
    if pre_mask.sum() < 3:
        # fallback: use first 10%
        pre_mask = t < (t.min() + 0.1*(t.max()-t.min()))

    post_end = min(t0 + 3.0, t.max())
    post_mask = (t >= t0) & (t <= post_end)
    if post_mask.sum() < 3:
        post_mask = t >= t0

    E_pre = np.median(E_t[pre_mask]) + 1e-12
    E_post = np.median(E_t[post_mask]) + 1e-12
    energy_jump_ratio = float(E_post / E_pre)

    # energy rise time: time from t0 until E exceeds (pre + 50%*(post-pre))
    target = E_pre + 0.5 * (E_post - E_pre)
    after = np.where((t >= t0) & (E_t >= target))[0]
    rise_time = float(t[after[0]] - t0) if len(after) else float("nan")

    # centroid slope during transient window (simple linear fit)
    ct = centroid_t[post_mask]
    tt = t[post_mask]
    if len(tt) >= 3:
        A = np.vstack([tt - tt.mean(), np.ones_like(tt)]).T
        slope, intercept = np.linalg.lstsq(A, ct, rcond=None)[0]
        centroid_slope = float(slope)
        centroid_mean = float(np.mean(ct))
    else:
        centroid_slope = float("nan")
        centroid_mean = float(np.mean(centroid_t))

    # band jump ratios
    band_jump = {}
    for bn, e in band_E_t.items():
        e = np.asarray(e)
        pre = np.median(e[pre_mask]) + 1e-12
        post = np.median(e[post_mask]) + 1e-12
        band_jump[f"band_jump_{bn}"] = float(post / pre)

    return {
        "t0": float(t0),
        "energy_jump_ratio": energy_jump_ratio,
        "rise_time_sec": rise_time,
        "centroid_mean_hz": centroid_mean,
        "centroid_slope_hz_per_s": centroid_slope,
        **band_jump
    }

def maybe_cwt_energy(x: np.ndarray, fs: float):
    if not DO_CWT or not HAS_PYWT:
        return None
    # CWT is slower; we compute scalogram power
    # Choose scales to cover ~1 Hz to ~5 kHz
    fmin, fmax = 5.0, min(5000.0, fs/2 - 10)
    if fmax <= fmin:
        return None
    # Convert desired freqs to scales for Morlet approx: scale ~ fs / freq
    freqs = np.geomspace(fmin, fmax, 64)
    scales = (fs / freqs)
    coeffs, _freqs = pywt.cwt(x, scales, CWT_WAVELET, sampling_period=1/fs)
    power = np.abs(coeffs) ** 2  # (scales,freq-like) x (time)
    return {"freqs": freqs, "power": power}

def plot_stft(meta: Meta, x: np.ndarray, fs: float, stft_pack: Dict[str, object], metrics: Dict[str, object], outpath: str):
    f, t, P = stft_pack["f"], stft_pack["t"], stft_pack["P"]
    E_t, centroid_t = stft_pack["E_t"], stft_pack["centroid_t"]

    plt.figure(figsize=(12, 8))
    plt.suptitle(f"{meta.sensor_group} | {meta.topology} {meta.leak} | Flow={meta.flow} | Noise={meta.noise} | {meta.sensor_id}", fontsize=12)

    # 1) waveform (downsample for speed)
    ax1 = plt.subplot(3, 1, 1)
    tt = np.arange(len(x)) / fs
    if len(tt) > 200000:
        idx = np.linspace(0, len(tt)-1, 200000).astype(int)
        ax1.plot(tt[idx], x[idx])
    else:
        ax1.plot(tt, x)
    ax1.set_ylabel("Amplitude")
    ax1.set_xlim([0, SIGNAL_DURATION_SEC])
    if "t0" in metrics:
        ax1.axvline(metrics["t0"], linestyle="--")
    ax1.grid(True, alpha=0.25)

    # 2) spectrogram (log power)
    ax2 = plt.subplot(3, 1, 2)
    Pdb = 10*np.log10(P + 1e-12)
    ax2.pcolormesh(t, f, Pdb, shading="gouraud")
    ax2.set_ylabel("Frequency (Hz)")
    ax2.set_ylim([0, min(5000, fs/2)])
    if "t0" in metrics:
        ax2.axvline(metrics["t0"], linestyle="--")
    ax2.grid(False)

    # 3) energy + centroid
    ax3 = plt.subplot(3, 1, 3)
    ax3.plot(t, np.log(E_t + 1e-12), label="log Energy")
    ax3.set_ylabel("log Energy")
    ax3.set_xlabel("Time (s)")
    if "t0" in metrics:
        ax3.axvline(metrics["t0"], linestyle="--", label="Transient start")
    ax3.grid(True, alpha=0.25)

    ax3b = ax3.twinx()
    ax3b.plot(t, centroid_t, label="Centroid", alpha=0.8)
    ax3b.set_ylabel("Spectral centroid (Hz)")

    # annotate key metrics
    txt = []
    for k in ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s"]:
        if k in metrics and np.isfinite(metrics[k]):
            txt.append(f"{k}={metrics[k]:.3g}")
    if txt:
        ax3.text(0.01, 0.95, "\n".join(txt), transform=ax3.transAxes, va="top")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(outpath, dpi=PLOT_DPI)
    plt.close()

# =========================
# ======= MAIN RUN ========
# =========================

# 1) Discover files
all_files = []
for root in ROOTS:
    if not os.path.exists(root):
        print(f"[WARN] Missing path: {root}")
        continue
    # accelerometer & pressure are .csv; hydrophone is .raw (plus maybe .wav etc)
    all_files += glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
    all_files += glob.glob(os.path.join(root, "**", "*.raw"), recursive=True)

if len(all_files) == 0:
    raise RuntimeError("No files found. Check ROOTS paths and file extensions.")

# 2) Process each file
rows = []
plot_count = 0

for path in tqdm(all_files, desc="Processing signals"):
    # Determine sensor group from path
    up = path.upper()
    if "ACCELEROMETER" in up:
        sensor_group = "Accelerometer"
    elif "DYNAMIC PRESSURE SENSOR" in up or "DYNAMIC_PRESSURE_SENSOR" in up:
        sensor_group = "Dynamic Pressure Sensor"
    elif "HYDROPHONE" in up:
        sensor_group = "Hydrophone"
    else:
        sensor_group = "Unknown"

    meta = parse_meta_from_filename(sensor_group, path)

    # Load signal
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv":
            x, fs = load_csv_signal(path)
        elif ext == ".raw":
            x, fs = load_raw_hydrophone(path, FS_HYDROPHONE)
        else:
            continue
    except Exception as e:
        rows.append({
            "path": path, "error": f"load_failed: {e}"
        })
        continue

    # Trim/pad to 30 s; basic preprocessing
    x = trim_or_pad(x, fs, SIGNAL_DURATION_SEC)
    x = x - np.mean(x)
    # light band-limit for numerical stability (doesn't “invent” info, just avoids DC/ultrasonic issues)
    try:
        # high-pass at 1 Hz
        sos = signal.butter(4, 1.0, btype="highpass", fs=fs, output="sos")
        x = signal.sosfiltfilt(sos, x)
    except Exception:
        pass

    # STFT features
    stft_pack = stft_energy_features(x, fs)

    # Decide transient start time t0:
    # 1) If filename indicates transient: set t0 = 0 (or detect anyway)
    # 2) Else: auto-detect if enabled; if no detection, choose mid as neutral reference
    is_transient_named = ("TRANSIENT" in meta.flow) or any(s in os.path.basename(path).upper() for s in ["TR_", "_TR_", "TRANS", "TRANSIENT"])
    if is_transient_named:
        detected = True
        # still try detect for better alignment if possible
        detected2, t0 = detect_transient_time(stft_pack["E_t"], stft_pack["t"])
        t0 = t0 if (AUTO_TRANSIENT_DETECT and detected2) else float(stft_pack["t"][0])  # onset
    else:
        if AUTO_TRANSIENT_DETECT:
            detected, t0 = detect_transient_time(stft_pack["E_t"], stft_pack["t"])
        else:
            detected, t0 = (False, float(stft_pack["t"][len(stft_pack["t"])//2]))

    metrics = propagation_metrics(
        stft_pack["E_t"],
        stft_pack["centroid_t"],
        stft_pack["band_E_t"],
        stft_pack["t"],
        t0
    )
    metrics["transient_detected"] = bool(detected)

    # Optional: CWT (not required; heavy)
    if DO_CWT:
        cwt_pack = maybe_cwt_energy(x, fs)
        if cwt_pack is not None:
            # summarize wavelet energy in same bands (approx mapping from freqs)
            freqs = cwt_pack["freqs"]
            power = cwt_pack["power"]
            # power shape: (n_freqs, n_time)
            # compute median power pre/post for each band
            # build a pseudo-time axis
            t_cwt = np.arange(power.shape[1]) / fs
            for bn, (lo, hi) in BANDS.items():
                idx = np.where((freqs >= max(lo, freqs.min())) & (freqs < min(hi, freqs.max()+1e-9)))[0]
                if len(idx) > 0:
                    Eband = np.sum(power[idx, :], axis=0)
                    # baseline vs transient
                    pre_mask = t_cwt < t0
                    if pre_mask.sum() < 10:
                        pre_mask = t_cwt < (t_cwt.min() + 0.1*(t_cwt.max()-t_cwt.min()))
                    post_mask = (t_cwt >= t0) & (t_cwt <= min(t0+3.0, t_cwt.max()))
                    if post_mask.sum() < 10:
                        post_mask = t_cwt >= t0
                    pre = np.median(Eband[pre_mask]) + 1e-12
                    post = np.median(Eband[post_mask]) + 1e-12
                    metrics[f"cwt_band_jump_{bn}"] = float(post/pre)

    # Save plot
    if MAKE_PLOTS and plot_count < MAX_PLOTS:
        # only plot a balanced subset (avoid 1000 plots). Prefer transient detected or transient-named.
        if detected or is_transient_named:
            plot_dir = safe_mkdir(os.path.join(OUT_DIR, "plots"))
            fname = f"{meta.sensor_group}_{meta.topology}_{meta.leak}_{meta.flow}_{meta.noise}_{meta.sensor_id}_{plot_count}.png"
            outp = os.path.join(plot_dir, re.sub(r"[^\w\-_\. ]", "_", fname))
            try:
                plot_stft(meta, x, fs, stft_pack, metrics, outp)
                plot_count += 1
            except Exception:
                pass

    # Row for CSV
    row = {
        "sensor_group": meta.sensor_group,
        "topology": meta.topology,
        "leak": meta.leak,
        "flow": meta.flow,
        "noise": meta.noise,
        "sensor_id": meta.sensor_id,
        "fs": fs,
        "path": meta.path,
        **metrics
    }
    rows.append(row)

# 3) Export results
res = pd.DataFrame(rows)

# Clean columns order
base_cols = ["sensor_group", "topology", "leak", "flow", "noise", "sensor_id", "fs",
             "transient_detected", "t0",
             "energy_jump_ratio", "rise_time_sec",
             "centroid_mean_hz", "centroid_slope_hz_per_s"]
band_cols = [c for c in res.columns if c.startswith("band_jump_")]
cwt_cols = [c for c in res.columns if c.startswith("cwt_band_jump_")]
other_cols = [c for c in res.columns if c not in set(base_cols + band_cols + cwt_cols + ["path", "error"])]

ordered = [c for c in base_cols if c in res.columns] + band_cols + cwt_cols + ["path"] + [c for c in other_cols if c in res.columns] + (["error"] if "error" in res.columns else [])
res = res[ordered]

out_csv = os.path.join(OUT_DIR, "transient_energy_summary.csv")
res.to_csv(out_csv, index=False)

# 4) Paper-ready summary tables
def summarize(df: pd.DataFrame, group_cols: List[str], metrics_cols: List[str]) -> pd.DataFrame:
    g = df.groupby(group_cols, dropna=False)
    agg = {}
    for m in metrics_cols:
        if m in df.columns:
            agg[m] = ["median", "mean", "std", "count"]
    out = g.agg(agg)
    out.columns = ["_".join(c).strip() for c in out.columns.to_flat_index()]
    out = out.reset_index()
    return out

metrics_cols = ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s"] + band_cols
summary1 = summarize(res.dropna(subset=["energy_jump_ratio"]), ["sensor_group", "topology", "leak", "flow"], metrics_cols)
summary2 = summarize(res.dropna(subset=["energy_jump_ratio"]), ["sensor_group", "leak"], metrics_cols)

summary1_path = os.path.join(OUT_DIR, "table_by_sensor_topology_leak_flow.csv")
summary2_path = os.path.join(OUT_DIR, "table_by_sensor_leak.csv")
summary1.to_csv(summary1_path, index=False)
summary2.to_csv(summary2_path, index=False)

# 5) Quick diagnostic plots (aggregate)
# Energy jump ratio by leak type, per sensor group (paper figure candidate)
try:
    figpath = os.path.join(OUT_DIR, "FIG_energy_jump_by_leak_sensor.png")
    plt.figure(figsize=(11, 5))
    dfp = res.dropna(subset=["energy_jump_ratio"]).copy()
    # keep reasonable finite
    dfp = dfp[np.isfinite(dfp["energy_jump_ratio"].astype(float))]
    # cap extreme ratios for visualization
    dfp["energy_jump_ratio_cap"] = np.clip(dfp["energy_jump_ratio"].astype(float), 0, np.nanquantile(dfp["energy_jump_ratio"].astype(float), 0.98))
    # boxplot-like using pandas
    order_leak = sorted(dfp["leak"].unique())
    order_sensor = sorted(dfp["sensor_group"].unique())
    # Create one subplot-like simple overlay by sensor (no subplots per instruction? Those instructions apply to python_user_visible; here local is fine.
    # But we'll do a single plot with grouped positions.
    pos = 0
    xticks, xlabels = [], []
    for sg in order_sensor:
        dsg = dfp[dfp["sensor_group"] == sg]
        for lk in order_leak:
            v = dsg[dsg["leak"] == lk]["energy_jump_ratio_cap"].values
            if len(v) == 0: 
                pos += 1
                continue
            plt.boxplot(v, positions=[pos], widths=0.6, showfliers=False)
            xticks.append(pos)
            xlabels.append(f"{sg[:4]}-{lk}")
            pos += 1
        pos += 1  # gap between sensor groups
    plt.xticks(xticks, xlabels, rotation=45, ha="right")
    plt.ylabel("Energy jump ratio (capped)")
    plt.title("Transient Energy Jump Ratio by Leak Type and Sensor (candidate MSSP figure)")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(figpath, dpi=PLOT_DPI)
    plt.close()
except Exception:
    pass

print("DONE ✅")
print(f"Saved main summary: {out_csv}")
print(f"Saved table (sensor+topology+leak+flow): {summary1_path}")
print(f"Saved table (sensor+leak): {summary2_path}")
plots_dir = os.path.join(OUT_DIR, "plots")
if os.path.exists(plots_dir):
    print(f"Saved plots (subset): {plots_dir}")
else:
    print("No plots saved (either disabled or none met criteria).")

# =========================
# HOW TO USE FOR YOUR PAPER
# =========================
# - Use transient_energy_summary.csv for modeling/analysis and paper stats.
# - Use FIG_energy_jump_by_leak_sensor.png as a candidate figure.
# - The 'plots' folder contains spectrogram+energy panels with detected transient line.
#
# If results look wrong:
# 1) Fix FS_HYDROPHONE (44100 vs 48000).
# 2) Fix FS_CSV_DEFAULT if time column missing.
# 3) Adjust BANDS and STFT_NPERSEG to match your sensor spectra.
# 4) If transient detection is too sensitive/insensitive, adjust TRANSIENT_DETECT_THRESHOLD_Z.


Processing signals: 100%|██████████| 282/282 [10:32<00:00,  2.24s/it]


DONE ✅
Saved main summary: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\transient_energy_summary.csv
Saved table (sensor+topology+leak+flow): E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\table_by_sensor_topology_leak_flow.csv
Saved table (sensor+leak): E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\table_by_sensor_leak.csv
Saved plots (subset): E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\plots


In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# ======= CONFIG ==========
# =========================
IN_CSV = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\transient_energy_summary.csv"
OUT_DIR = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures"
os.makedirs(OUT_DIR, exist_ok=True)

# Controls
CAP_Q = 0.98     # cap extreme ratios for plotting clarity
MIN_COUNT = 5    # minimum samples per group to include in plots/tables

# =========================
# ======= LOAD ============
# =========================
df = pd.read_csv(IN_CSV)

# Basic cleaning
for c in ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s",
          "band_jump_low", "band_jump_mid", "band_jump_high"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Remove rows with load errors if any
if "error" in df.columns:
    df = df[df["error"].isna()].copy()

# Define leak order (nice for figures)
leak_order_preferred = ["NL", "OL", "GL", "LC", "CC"]  # No-leak, Orifice, Gasket, Longitudinal, Circumferential
leaks_present = [l for l in leak_order_preferred if l in set(df["leak"].astype(str))]
# Add any unexpected leak labels at the end
leaks_present += [l for l in sorted(df["leak"].dropna().unique()) if l not in leaks_present]

sensor_order = sorted(df["sensor_group"].dropna().unique())
topo_order = sorted(df["topology"].dropna().unique())

# =========================
# ======= SUBSETS =========
# =========================
# Main analysis (reviewer-proof): labeled transient
df_main = df[df["flow"].astype(str).str.contains("TRANSIENT", case=False, na=False)].copy()

# Sensitivity check: any detected transient
df_sens = df[df["transient_detected"] == True].copy()

# If your filenames didn't encode "TRANSIENT", df_main may be empty.
# In that case, we fallback to sensitivity as main, but we will print a warning.
if len(df_main) < 10:
    print("[WARN] Very few labeled TRANSIENT rows found. Your filenames may not include TRANSIENT tokens.")
    print("       I will use transient_detected=True as MAIN for now, and save labeled-transient as sensitivity.")
    df_main, df_sens = df_sens, df_main

print(f"Main subset rows: {len(df_main)}")
print(f"Sensitivity subset rows: {len(df_sens)}")

# =========================
# ======= HELPERS =========
# =========================
def cap_series(s, q=0.98):
    s = s.astype(float)
    s = s[np.isfinite(s)]
    if len(s) == 0:
        return s
    cap = np.nanquantile(s, q)
    return np.clip(s, -cap, cap)

def group_stats(dfx, group_cols, metrics):
    out = dfx.groupby(group_cols).agg({m: ["count", "median", "mean", "std"] for m in metrics if m in dfx.columns})
    out.columns = ["_".join(x) for x in out.columns.to_flat_index()]
    out = out.reset_index()
    # filter small groups
    if any(f"{metrics[0]}_count" in out.columns for _ in [0]):
        # if metrics[0] exists; otherwise skip
        pass
    return out

def make_boxplot_by_sensor(dfx, metric, title, fname, leaks=leaks_present, sensors=sensor_order, cap_q=CAP_Q):
    if metric not in dfx.columns:
        print(f"[SKIP] Missing metric: {metric}")
        return

    # prepare capped values for plotting
    d = dfx.dropna(subset=[metric, "leak", "sensor_group"]).copy()
    d = d[np.isfinite(d[metric].astype(float))]
    if len(d) < 10:
        print(f"[SKIP] Not enough data for {metric}")
        return

    # cap per entire dataset for consistency
    cap = np.nanquantile(d[metric].astype(float), cap_q)
    d["metric_cap"] = np.clip(d[metric].astype(float), -cap, cap)

    positions = []
    data = []
    labels = []

    pos = 0
    xticks = []
    xlabels = []

    for sg in sensors:
        ds = d[d["sensor_group"] == sg]
        for lk in leaks:
            vals = ds[ds["leak"] == lk]["metric_cap"].values
            if len(vals) >= MIN_COUNT:
                data.append(vals)
                positions.append(pos)
                xticks.append(pos)
                xlabels.append(f"{sg[:4]}-{lk}")
            pos += 1
        pos += 1  # gap

    if len(data) == 0:
        print(f"[SKIP] No groups meet MIN_COUNT={MIN_COUNT} for {metric}")
        return

    plt.figure(figsize=(12, 5))
    plt.boxplot(data, positions=positions, widths=0.6, showfliers=False)
    plt.xticks(xticks, xlabels, rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(title)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    outpath = os.path.join(OUT_DIR, fname)
    plt.savefig(outpath, dpi=180)
    plt.close()
    print(f"Saved: {outpath}")

def make_band_profile_plot(dfx, title, fname, leaks=leaks_present, sensors=sensor_order):
    needed = ["band_jump_low", "band_jump_mid", "band_jump_high"]
    if not all(c in dfx.columns for c in needed):
        print("[SKIP] Band jumps missing.")
        return

    d = dfx.dropna(subset=needed + ["leak", "sensor_group"]).copy()
    if len(d) < 10:
        print("[SKIP] Not enough data for band profile.")
        return

    # cap ratios for plot clarity
    for c in needed:
        cap = np.nanquantile(d[c].astype(float), CAP_Q)
        d[c] = np.clip(d[c].astype(float), 0, cap)

    # median band profile per sensor+leak
    g = d.groupby(["sensor_group", "leak"])[needed].median().reset_index()

    # plot as grouped bars on one figure: for each sensor group create a block; within each leak, show 3 bars
    plt.figure(figsize=(13, 5))
    x = 0
    xticks = []
    xlabels = []

    width = 0.22
    for sg in sensors:
        gs = g[g["sensor_group"] == sg]
        for lk in leaks:
            row = gs[gs["leak"] == lk]
            if len(row) == 0:
                x += 1
                continue
            low, mid, high = row[needed].values[0]
            # 3 bars around x
            plt.bar(x - width, low, width=width)
            plt.bar(x,         mid, width=width)
            plt.bar(x + width, high, width=width)
            xticks.append(x)
            xlabels.append(f"{sg[:4]}-{lk}")
            x += 1
        x += 1  # gap

    if len(xticks) == 0:
        print("[SKIP] No band-profile groups.")
        return

    plt.xticks(xticks, xlabels, rotation=45, ha="right")
    plt.ylabel("Median band jump ratio (capped)")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.25)
    plt.tight_layout()
    outpath = os.path.join(OUT_DIR, fname)
    plt.savefig(outpath, dpi=180)
    plt.close()
    print(f"Saved: {outpath}")

# =========================
# ======= FIGURES =========
# =========================
# MAIN FIGURES (these are what you put in paper)
make_boxplot_by_sensor(
    df_main, "energy_jump_ratio",
    "MAIN (Labeled Transient): Energy jump ratio by leak type and sensor",
    "FIG_MAIN_energy_jump_ratio.png"
)

make_band_profile_plot(
    df_main,
    "MAIN (Labeled Transient): Median band-energy jump profile (low/mid/high) by leak type and sensor",
    "FIG_MAIN_band_jump_profile.png"
)

make_boxplot_by_sensor(
    df_main, "rise_time_sec",
    "MAIN (Labeled Transient): Rise time (s) by leak type and sensor",
    "FIG_MAIN_rise_time.png"
)

make_boxplot_by_sensor(
    df_main, "centroid_slope_hz_per_s",
    "MAIN (Labeled Transient): Spectral centroid slope (Hz/s) by leak type and sensor",
    "FIG_MAIN_centroid_slope.png"
)

# SENSITIVITY FIGURES (appendix / supplementary)
if len(df_sens) >= 10:
    make_boxplot_by_sensor(
        df_sens, "energy_jump_ratio",
        "SENSITIVITY (Detected Transient): Energy jump ratio by leak type and sensor",
        "FIG_SENS_energy_jump_ratio.png"
    )

    make_band_profile_plot(
        df_sens,
        "SENSITIVITY (Detected Transient): Median band-energy jump profile by leak type and sensor",
        "FIG_SENS_band_jump_profile.png"
    )

# =========================
# ======= TABLES ==========
# =========================
metrics_for_tables = ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s",
                      "band_jump_low", "band_jump_mid", "band_jump_high"]

tab_main = group_stats(df_main, ["sensor_group", "topology", "leak", "flow"], metrics_for_tables)
tab_main_path = os.path.join(OUT_DIR, "TABLE_MAIN_sensor_topology_leak_flow_stats.csv")
tab_main.to_csv(tab_main_path, index=False)
print(f"Saved: {tab_main_path}")

tab_main2 = group_stats(df_main, ["sensor_group", "leak"], metrics_for_tables)
tab_main2_path = os.path.join(OUT_DIR, "TABLE_MAIN_sensor_leak_stats.csv")
tab_main2.to_csv(tab_main2_path, index=False)
print(f"Saved: {tab_main2_path}")

if len(df_sens) >= 10:
    tab_sens = group_stats(df_sens, ["sensor_group", "leak"], metrics_for_tables)
    tab_sens_path = os.path.join(OUT_DIR, "TABLE_SENS_sensor_leak_stats.csv")
    tab_sens.to_csv(tab_sens_path, index=False)
    print(f"Saved: {tab_sens_path}")

print("\nDONE ✅ Paper figures + tables saved to:")
print(OUT_DIR)


Main subset rows: 80
Sensitivity subset rows: 183
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\FIG_MAIN_energy_jump_ratio.png
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\FIG_MAIN_band_jump_profile.png
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\FIG_MAIN_rise_time.png
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\FIG_MAIN_centroid_slope.png
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\FIG_SENS_energy_jump_ratio.png
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\FIG_SENS_band_jump_profile.png
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\TABLE_MAIN_sensor_topology_leak_flow_stats.csv
Saved: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\paper_figures\TABLE_MA

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

# =========================
# ====== GLOBAL STYLE =====
# =========================
rcParams["font.family"] = "serif"
rcParams["font.size"] = 14
rcParams["axes.labelweight"] = "bold"
rcParams["axes.titleweight"] = "bold"
rcParams["axes.linewidth"] = 1.8
rcParams["xtick.major.width"] = 1.5
rcParams["ytick.major.width"] = 1.5
rcParams["legend.frameon"] = False

# =========================
# ====== PATH CONFIG ======
# =========================
ROOT = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy"
CSV = os.path.join(ROOT, "transient_energy_summary.csv")
FIGDIR = os.path.join(ROOT, "MSSP_FIGURES")
os.makedirs(FIGDIR, exist_ok=True)

df = pd.read_csv(CSV)

# Clean
metrics = [
    "energy_jump_ratio",
    "rise_time_sec",
    "centroid_slope_hz_per_s",
    "band_jump_low",
    "band_jump_mid",
    "band_jump_high"
]
for m in metrics:
    df[m] = pd.to_numeric(df[m], errors="coerce")

df = df[df["error"].isna()] if "error" in df.columns else df

# =========================
# ====== SUBSETS ==========
# =========================
df_main = df[df["flow"].str.contains("TRANSIENT", na=False)]
df_sens = df[df["transient_detected"] == True]

leak_order = ["NL", "OL", "GL", "LC", "CC"]
sensor_order = ["Accelerometer", "Dynamic Pressure Sensor", "Hydrophone"]

# =========================
# ====== HELPERS ==========
# =========================
def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIGDIR, name), dpi=300)
    plt.close()

def boxplot(metric, title, fname, data):
    plt.figure(figsize=(14, 5))
    pos = 0
    xticks, labels = [], []
    for s in sensor_order:
        ds = data[data["sensor_group"] == s]
        for l in leak_order:
            vals = ds[ds["leak"] == l][metric].dropna()
            if len(vals) > 5:
                plt.boxplot(
                    np.clip(vals, 0, np.nanquantile(vals, 0.98)),
                    positions=[pos],
                    widths=0.6,
                    showfliers=False
                )
                xticks.append(pos)
                labels.append(f"{s.split()[0]}-{l}")
            pos += 1
        pos += 1
    plt.xticks(xticks, labels, rotation=45, ha="right")
    plt.ylabel(metric.replace("_", " ").title())
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    savefig(fname)

def band_profile(title, fname, data):
    plt.figure(figsize=(14, 5))
    width = 0.25
    x = 0
    xticks, labels = [], []
    for s in sensor_order:
        ds = data[data["sensor_group"] == s]
        for l in leak_order:
            row = ds[ds["leak"] == l]
            if len(row) == 0:
                x += 1
                continue
            low = np.nanmedian(row["band_jump_low"])
            mid = np.nanmedian(row["band_jump_mid"])
            high = np.nanmedian(row["band_jump_high"])
            plt.bar(x - width, low, width)
            plt.bar(x, mid, width)
            plt.bar(x + width, high, width)
            xticks.append(x)
            labels.append(f"{s.split()[0]}-{l}")
            x += 1
        x += 1
    plt.xticks(xticks, labels, rotation=45, ha="right")
    plt.ylabel("Median Band Energy Jump Ratio")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    savefig(fname)

# =========================
# ====== MAIN FIGURES =====
# =========================
boxplot(
    "energy_jump_ratio",
    "FIG.5  Transient Energy Jump Ratio by Leak Type and Sensor",
    "FIG5_energy_jump_ratio_MAIN.png",
    df_main
)

band_profile(
    "FIG.6  Band-Specific Energy Propagation Profiles",
    "FIG6_band_energy_profile_MAIN.png",
    df_main
)

boxplot(
    "rise_time_sec",
    "FIG.7  Transient Energy Rise Time Distribution",
    "FIG7_rise_time_MAIN.png",
    df_main
)

boxplot(
    "centroid_slope_hz_per_s",
    "FIG.8  Spectral Centroid Evolution During Transients",
    "FIG8_centroid_slope_MAIN.png",
    df_main
)

# =========================
# == SUPPLEMENTARY FIGURES =
# =========================
boxplot(
    "energy_jump_ratio",
    "S1  Sensitivity: Energy Jump Ratio (Detected Transients)",
    "S1_energy_jump_ratio_SENS.png",
    df_sens
)

band_profile(
    "S2  Sensitivity: Band Energy Profile (Detected Transients)",
    "S2_band_energy_profile_SENS.png",
    df_sens
)

print("\nALL MSSP-LEVEL FIGURES GENERATED SUCCESSFULLY ✅")
print("Saved in:", FIGDIR)



ALL MSSP-LEVEL FIGURES GENERATED SUCCESSFULLY ✅
Saved in: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_FIGURES


In [10]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

# =========================
# ====== GLOBAL STYLE =====
# =========================
rcParams["font.family"] = "serif"
rcParams["font.size"] = 14
rcParams["axes.labelweight"] = "bold"
rcParams["axes.titleweight"] = "bold"
rcParams["axes.linewidth"] = 1.8
rcParams["xtick.major.width"] = 1.5
rcParams["ytick.major.width"] = 1.5
rcParams["legend.frameon"] = False

# =========================
# ====== PATH CONFIG ======
# =========================
ROOT = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy"
CSV = os.path.join(ROOT, "transient_energy_summary.csv")
FIGDIR = os.path.join(ROOT, "MSSP_FIGURES")
os.makedirs(FIGDIR, exist_ok=True)

df = pd.read_csv(CSV)

# Clean
metrics = [
    "energy_jump_ratio",
    "rise_time_sec",
    "centroid_slope_hz_per_s",
    "band_jump_low",
    "band_jump_mid",
    "band_jump_high"
]
for m in metrics:
    df[m] = pd.to_numeric(df[m], errors="coerce")

df = df[df["error"].isna()] if "error" in df.columns else df

# =========================
# ====== SUBSETS ==========
# =========================
df_main = df[df["flow"].str.contains("TRANSIENT", na=False)]
df_sens = df[df["transient_detected"] == True]

leak_order = ["NL", "OL", "GL", "LC", "CC"]
sensor_order = ["Accelerometer", "Dynamic Pressure Sensor", "Hydrophone"]

# =========================
# ====== HELPERS ==========
# =========================
def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIGDIR, name), dpi=300)
    plt.close()

def boxplot(metric, title, fname, data):
    plt.figure(figsize=(14, 5))
    pos = 0
    xticks, labels = [], []
    for s in sensor_order:
        ds = data[data["sensor_group"] == s]
        for l in leak_order:
            vals = ds[ds["leak"] == l][metric].dropna()
            if len(vals) > 5:
                plt.boxplot(
                    np.clip(vals, 0, np.nanquantile(vals, 0.98)),
                    positions=[pos],
                    widths=0.6,
                    showfliers=False
                )
                xticks.append(pos)
                labels.append(f"{s.split()[0]}-{l}")
            pos += 1
        pos += 1
    plt.xticks(xticks, labels, rotation=45, ha="right")
    plt.ylabel(metric.replace("_", " ").title())
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    savefig(fname)

def band_profile(title, fname, data):
    plt.figure(figsize=(14, 5))
    width = 0.25
    x = 0
    xticks, labels = [], []
    for s in sensor_order:
        ds = data[data["sensor_group"] == s]
        for l in leak_order:
            row = ds[ds["leak"] == l]
            if len(row) == 0:
                x += 1
                continue
            low = np.nanmedian(row["band_jump_low"])
            mid = np.nanmedian(row["band_jump_mid"])
            high = np.nanmedian(row["band_jump_high"])
            plt.bar(x - width, low, width)
            plt.bar(x, mid, width)
            plt.bar(x + width, high, width)
            xticks.append(x)
            labels.append(f"{s.split()[0]}-{l}")
            x += 1
        x += 1
    plt.xticks(xticks, labels, rotation=45, ha="right")
    plt.ylabel("Median Band Energy Jump Ratio")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    savefig(fname)

# =========================
# ====== MAIN FIGURES =====
# =========================
boxplot(
    "energy_jump_ratio",
    "FIG.5  Transient Energy Jump Ratio by Leak Type and Sensor",
    "FIG5_energy_jump_ratio_MAIN.png",
    df_main
)

band_profile(
    "FIG.6  Band-Specific Energy Propagation Profiles",
    "FIG6_band_energy_profile_MAIN.png",
    df_main
)

boxplot(
    "rise_time_sec",
    "FIG.7  Transient Energy Rise Time Distribution",
    "FIG7_rise_time_MAIN.png",
    df_main
)

boxplot(
    "centroid_slope_hz_per_s",
    "FIG.8  Spectral Centroid Evolution During Transients",
    "FIG8_centroid_slope_MAIN.png",
    df_main
)

# =========================
# == SUPPLEMENTARY FIGURES =
# =========================
boxplot(
    "energy_jump_ratio",
    "S1  Sensitivity: Energy Jump Ratio (Detected Transients)",
    "S1_energy_jump_ratio_SENS.png",
    df_sens
)

band_profile(
    "S2  Sensitivity: Band Energy Profile (Detected Transients)",
    "S2_band_energy_profile_SENS.png",
    df_sens
)

print("\nALL MSSP-LEVEL FIGURES GENERATED SUCCESSFULLY ✅")
print("Saved in:", FIGDIR)



ALL MSSP-LEVEL FIGURES GENERATED SUCCESSFULLY ✅
Saved in: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_FIGURES


In [11]:
# MASSIVE MSSP-QUALITY FIGURE GENERATOR (50+ figures, clear legends, concise, publication-ready)
# Uses ONLY your computed summary CSV (fast, robust, reproducible).
#
# Output: 100+ high-quality PNG figures organized into folders (MAIN vs SUPP, metric vs grouping).
#
# What you get:
# - For each metric (energy_jump_ratio, rise_time_sec, centroid_slope_hz_per_s, band_jump_low/mid/high):
#   (A) Sensor-wise leak comparisons (boxplots)
#   (B) Leak-wise sensor comparisons (boxplots)
#   (C) Topology-separated plots (BR vs LO) (boxplots)
#   (D) Flow-separated plots (if multiple transient flavors exist)
#   (E) Per-sensor per-leak distributions (each as its own clean figure)
# - Band profile plots (low/mid/high) for each sensor, each topology, and each leak (bar/line variants)
# - Sensitivity (detected transient) versions of all key plot families
#
# Expect: 50+ figures easily (usually 120+).
#
# Run as ONE CELL.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

# =========================
# ========== CONFIG =======
# =========================
ROOT = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy"
IN_CSV = os.path.join(ROOT, "transient_energy_summary.csv")
OUT = os.path.join(ROOT, "MSSP_50PLUS_FIGURES")
os.makedirs(OUT, exist_ok=True)

DPI = 350
MIN_N = 5
CAP_Q = 0.98  # cap heavy tails for visual clarity
LEAK_ORDER_PREF = ["NL", "OL", "GL", "LC", "CC"]  # dataset acronyms
SENSOR_ORDER_PREF = ["Accelerometer", "Dynamic Pressure Sensor", "Hydrophone"]
TOPO_ORDER_PREF = ["BR", "LO"]  # typical
SHOWFliers = False  # MSSP style tends to hide outliers in main boxplots

# =========================
# ======= STYLE (MSSP) =====
# =========================
rcParams["font.family"] = "serif"
rcParams["font.size"] = 13
rcParams["axes.labelweight"] = "bold"
rcParams["axes.titleweight"] = "bold"
rcParams["axes.linewidth"] = 1.8
rcParams["xtick.major.width"] = 1.5
rcParams["ytick.major.width"] = 1.5
rcParams["legend.frameon"] = False
rcParams["figure.facecolor"] = "white"

def safe_mkdir(p):
    os.makedirs(p, exist_ok=True)
    return p

def finite_series(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s[np.isfinite(s)]
    return s

def cap_vals(vals, q=CAP_Q):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return vals
    cap = np.nanquantile(vals, q)
    # cap only upper for ratios (non-negative), but allow slopes negative
    return cap

def tight_save(path):
    plt.tight_layout()
    plt.savefig(path, dpi=DPI)
    plt.close()

def get_order(present, preferred):
    out = [x for x in preferred if x in present]
    out += [x for x in sorted(present) if x not in out]
    return out

# =========================
# ======= LOAD & CLEAN =====
# =========================
df = pd.read_csv(IN_CSV)

if "error" in df.columns:
    df = df[df["error"].isna()].copy()

# Numeric conversions
metrics = [
    "energy_jump_ratio",
    "rise_time_sec",
    "centroid_slope_hz_per_s",
    "band_jump_low",
    "band_jump_mid",
    "band_jump_high",
]
for m in metrics:
    if m in df.columns:
        df[m] = pd.to_numeric(df[m], errors="coerce")

# Basic availability
required_cols = {"sensor_group", "leak", "topology", "flow", "transient_detected"}
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required columns in CSV: {missing}")

# MAIN = labeled transient (reviewer-proof)
df_main = df[df["flow"].astype(str).str.contains("TRANSIENT", case=False, na=False)].copy()
# SENS = detected transient (sensitivity/appendix)
df_sens = df[df["transient_detected"] == True].copy()

if len(df_main) < 10:
    print("[WARN] Few labeled transient rows found; using detected transients as MAIN and labeled as SENS.")
    df_main, df_sens = df_sens, df_main

leaks = get_order(set(df["leak"].dropna().astype(str)), LEAK_ORDER_PREF)
sensors = get_order(set(df["sensor_group"].dropna().astype(str)), SENSOR_ORDER_PREF)
topos = get_order(set(df["topology"].dropna().astype(str)), TOPO_ORDER_PREF)
flows_main = sorted(set(df_main["flow"].dropna().astype(str)))
flows_sens = sorted(set(df_sens["flow"].dropna().astype(str)))

print(f"MAIN rows: {len(df_main)} | SENS rows: {len(df_sens)}")
print(f"Leaks: {leaks}")
print(f"Sensors: {sensors}")
print(f"Topologies: {topos}")
print(f"MAIN flows: {flows_main}")

# =========================
# ====== PLOTTING CORE =====
# =========================
def boxplot_grouped(data, metric, groupA, groupB, orderA, orderB, title, ylabel, outpath,
                    cap_upper=True, showfliers=SHOWFliers):
    """
    Create a boxplot for metric grouped by groupA blocks with groupB inside each block.
    Example: groupA='sensor_group', groupB='leak' => sensor blocks, leak per sensor.
    """
    d = data.dropna(subset=[metric, groupA, groupB]).copy()
    d = d[np.isfinite(d[metric].astype(float))]
    if len(d) < 10:
        return False

    # cap values for readability
    vals_all = d[metric].astype(float).values
    cap = np.nanquantile(vals_all, CAP_Q)
    if cap_upper:
        d["_m"] = np.clip(d[metric].astype(float), -cap if not cap_upper else 0, cap)
    else:
        d["_m"] = np.clip(d[metric].astype(float), -cap, cap)

    positions, series_list, xticks, xlabels = [], [], [], []
    pos = 0
    gap = 1

    for a in orderA:
        da = d[d[groupA] == a]
        for b in orderB:
            v = da[da[groupB] == b]["_m"].values
            if len(v) >= MIN_N:
                series_list.append(v)
                positions.append(pos)
                xticks.append(pos)
                # concise labels
                a_short = str(a).replace("Dynamic Pressure Sensor", "Pressure").replace("Accelerometer", "Accel").replace("Hydrophone", "Hydro")
                b_short = str(b)
                xlabels.append(f"{a_short}\n{b_short}")
            pos += 1
        pos += gap

    if len(series_list) == 0:
        return False

    plt.figure(figsize=(14, 5))
    plt.boxplot(series_list, positions=positions, widths=0.6, showfliers=showfliers)
    plt.xticks(xticks, xlabels, rotation=0, ha="center")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.25)
    tight_save(outpath)
    return True

def per_group_distribution(data, metric, filter_col, filter_val, title, outpath,
                           cap_upper=True, bins=30):
    """
    Single-group histogram + KDE-like line (via hist density) for a clean distribution plot.
    """
    d = data[data[filter_col] == filter_val].dropna(subset=[metric]).copy()
    v = finite_series(d[metric])
    if len(v) < MIN_N:
        return False

    cap = np.nanquantile(v, CAP_Q)
    if cap_upper:
        v = np.clip(v, 0, cap)
    else:
        v = np.clip(v, -cap, cap)

    plt.figure(figsize=(7.5, 4.5))
    plt.hist(v, bins=bins, density=True, alpha=0.8, edgecolor="black", linewidth=0.8)
    plt.xlabel(metric.replace("_", " ").title())
    plt.ylabel("Density")
    plt.title(title)
    plt.grid(True, alpha=0.2)
    tight_save(outpath)
    return True

def band_profile_median(data, title, outpath, by=("sensor_group","leak"), orderA=None, orderB=None):
    """
    Median low/mid/high band jump ratios in a bar group per (A,B).
    """
    needed = ["band_jump_low", "band_jump_mid", "band_jump_high"]
    d = data.dropna(subset=list(by) + needed).copy()
    for c in needed:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=needed)
    if len(d) < 10:
        return False

    # cap for readability
    for c in needed:
        cap = np.nanquantile(d[c].astype(float), CAP_Q)
        d[c] = np.clip(d[c].astype(float), 0, cap)

    A, B = by
    g = d.groupby([A,B])[needed].median().reset_index()

    if orderA is None:
        orderA = sorted(g[A].unique())
    if orderB is None:
        orderB = sorted(g[B].unique())

    plt.figure(figsize=(15, 5))
    width = 0.22
    x = 0
    xticks, xlabels = [], []

    # create a legend once using dummy bars
    plt.bar(-10, 0, width=width, label="Low band")
    plt.bar(-10, 0, width=width, label="Mid band")
    plt.bar(-10, 0, width=width, label="High band")

    for a in orderA:
        ga = g[g[A] == a]
        for b in orderB:
            row = ga[ga[B] == b]
            if len(row) == 0:
                x += 1
                continue
            low, mid, high = row[needed].values[0]
            plt.bar(x - width, low, width=width)
            plt.bar(x,         mid, width=width)
            plt.bar(x + width, high, width=width)
            xticks.append(x)
            a_short = str(a).replace("Dynamic Pressure Sensor", "Pressure").replace("Accelerometer", "Accel").replace("Hydrophone", "Hydro")
            xlabels.append(f"{a_short}\n{b}")
            x += 1
        x += 1  # gap between A blocks

    plt.xticks(xticks, xlabels, rotation=0, ha="center")
    plt.ylabel("Median Band Energy Jump Ratio (capped)")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.25)
    plt.legend(ncol=3, loc="upper right")
    tight_save(outpath)
    return True

# =========================
# ===== FIGURE FACTORY =====
# =========================
main_dir = safe_mkdir(os.path.join(OUT, "MAIN"))
supp_dir = safe_mkdir(os.path.join(OUT, "SUPPLEMENTARY"))

def generate_all_for_dataset(tag, data, base_dir):
    count = 0

    # 1) Core boxplots: Sensor->Leak for each metric
    core_dir = safe_mkdir(os.path.join(base_dir, "A_core_boxplots"))
    for m in ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s", "band_jump_low", "band_jump_mid", "band_jump_high"]:
        if m not in data.columns:
            continue
        cap_upper = (m != "centroid_slope_hz_per_s")  # slopes can be negative
        ok = boxplot_grouped(
            data, m,
            groupA="sensor_group", groupB="leak",
            orderA=sensors, orderB=leaks,
            title=f"{tag}: {m.replace('_',' ').title()} (by Sensor → Leak)",
            ylabel=m.replace("_"," ").title(),
            outpath=os.path.join(core_dir, f"{tag}_BOX_sensor_leak_{m}.png"),
            cap_upper=cap_upper
        )
        if ok: count += 1

    # 2) Core boxplots: Leak->Sensor for each metric
    core2_dir = safe_mkdir(os.path.join(base_dir, "B_core_boxplots_leak_sensor"))
    for m in ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s", "band_jump_low", "band_jump_mid", "band_jump_high"]:
        if m not in data.columns:
            continue
        cap_upper = (m != "centroid_slope_hz_per_s")
        ok = boxplot_grouped(
            data, m,
            groupA="leak", groupB="sensor_group",
            orderA=leaks, orderB=sensors,
            title=f"{tag}: {m.replace('_',' ').title()} (by Leak → Sensor)",
            ylabel=m.replace("_"," ").title(),
            outpath=os.path.join(core2_dir, f"{tag}_BOX_leak_sensor_{m}.png"),
            cap_upper=cap_upper
        )
        if ok: count += 1

    # 3) Topology-separated boxplots (BR and LO) for key metrics
    topo_dir = safe_mkdir(os.path.join(base_dir, "C_topology_separated"))
    key_metrics = ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s"]
    for topo in topos:
        dt = data[data["topology"].astype(str) == str(topo)]
        if len(dt) < 10:
            continue
        for m in key_metrics:
            if m not in dt.columns:
                continue
            cap_upper = (m != "centroid_slope_hz_per_s")
            ok = boxplot_grouped(
                dt, m,
                groupA="sensor_group", groupB="leak",
                orderA=sensors, orderB=leaks,
                title=f"{tag}: {m.replace('_',' ').title()} (Topology={topo})",
                ylabel=m.replace("_"," ").title(),
                outpath=os.path.join(topo_dir, f"{tag}_TOPO_{topo}_sensor_leak_{m}.png"),
                cap_upper=cap_upper
            )
            if ok: count += 1

    # 4) Flow-separated boxplots (if multiple flow tokens exist)
    flow_dir = safe_mkdir(os.path.join(base_dir, "D_flow_separated"))
    flows = sorted(set(data["flow"].dropna().astype(str)))
    # avoid too many if flow labels are messy; still, this is what you asked for
    for fl in flows:
        df_fl = data[data["flow"].astype(str) == fl]
        if len(df_fl) < 10:
            continue
        for m in ["energy_jump_ratio", "rise_time_sec"]:
            if m not in df_fl.columns:
                continue
            ok = boxplot_grouped(
                df_fl, m,
                groupA="sensor_group", groupB="leak",
                orderA=sensors, orderB=leaks,
                title=f"{tag}: {m.replace('_',' ').title()} (Flow={fl})",
                ylabel=m.replace("_"," ").title(),
                outpath=os.path.join(flow_dir, f"{tag}_FLOW_{fl.replace(' ','_')}_sensor_leak_{m}.png"),
                cap_upper=True
            )
            if ok: count += 1

    # 5) Band profile figures (median low/mid/high) in multiple groupings
    band_dir = safe_mkdir(os.path.join(base_dir, "E_band_profiles"))
    ok = band_profile_median(
        data,
        title=f"{tag}: Median Band Profile (Sensor → Leak)",
        outpath=os.path.join(band_dir, f"{tag}_BANDPROFILE_sensor_leak.png"),
        by=("sensor_group","leak"),
        orderA=sensors,
        orderB=leaks
    )
    if ok: count += 1

    # band profile per topology
    for topo in topos:
        dt = data[data["topology"].astype(str) == str(topo)]
        if len(dt) < 10:
            continue
        ok = band_profile_median(
            dt,
            title=f"{tag}: Median Band Profile (Sensor → Leak), Topology={topo}",
            outpath=os.path.join(band_dir, f"{tag}_BANDPROFILE_sensor_leak_TOPO_{topo}.png"),
            by=("sensor_group","leak"),
            orderA=sensors,
            orderB=leaks
        )
        if ok: count += 1

    # band profile per sensor (leak-only x-axis)
    for s in sensors:
        ds = data[data["sensor_group"].astype(str) == str(s)]
        if len(ds) < 10:
            continue
        ok = band_profile_median(
            ds,
            title=f"{tag}: Median Band Profile by Leak (Sensor={s})",
            outpath=os.path.join(band_dir, f"{tag}_BANDPROFILE_leak_SENSOR_{s.replace(' ','_')}.png"),
            by=("leak","sensor_group"),
            orderA=leaks,
            orderB=[s]
        )
        if ok: count += 1

    # 6) Per-sensor-per-leak distributions for key metrics (THIS ALONE gives 3*5*3 = 45+ figs)
    dist_dir = safe_mkdir(os.path.join(base_dir, "F_distributions_sensor_leak"))
    dist_metrics = ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s"]
    for s in sensors:
        ds = data[data["sensor_group"].astype(str) == str(s)]
        for l in leaks:
            dl = ds[ds["leak"].astype(str) == str(l)]
            if len(dl) < MIN_N:
                continue
            for m in dist_metrics:
                if m not in dl.columns:
                    continue
                cap_upper = (m != "centroid_slope_hz_per_s")
                title = f"{tag}: {m.replace('_',' ').title()} | Sensor={s} | Leak={l} (n={len(dl)})"
                outpath = os.path.join(dist_dir, f"{tag}_DIST_{m}_S_{s.replace(' ','_')}_L_{l}.png")
                ok = per_group_distribution(dl, m, filter_col="leak", filter_val=l, title=title, outpath=outpath,
                                            cap_upper=cap_upper, bins=28)
                if ok: count += 1

    # 7) Quick “summary heatmaps” as figures (median metric by sensor x leak)
    # (No seaborn; pure matplotlib)
    heat_dir = safe_mkdir(os.path.join(base_dir, "G_heatmaps"))
    def heatmap_median(metric, cap_upper=True):
        d = data.dropna(subset=["sensor_group","leak",metric]).copy()
        d = d[np.isfinite(d[metric].astype(float))]
        if len(d) < 10:
            return False
        # pivot median
        piv = d.pivot_table(index="sensor_group", columns="leak", values=metric, aggfunc="median")
        piv = piv.reindex(index=sensors, columns=leaks)
        arr = piv.to_numpy(dtype=float)
        if np.all(~np.isfinite(arr)):
            return False
        # cap for contrast
        finite = arr[np.isfinite(arr)]
        cap = np.nanquantile(finite, CAP_Q)
        if cap_upper:
            arr = np.clip(arr, 0, cap)
        else:
            arr = np.clip(arr, -cap, cap)

        plt.figure(figsize=(7.8, 4.8))
        im = plt.imshow(arr, aspect="auto")
        plt.xticks(range(len(leaks)), leaks)
        s_short = [x.replace("Dynamic Pressure Sensor","Pressure").replace("Accelerometer","Accel").replace("Hydrophone","Hydro") for x in sensors]
        plt.yticks(range(len(sensors)), s_short)
        plt.title(f"{tag}: Median {metric.replace('_',' ').title()} (Sensor × Leak)")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        plt.tight_layout()
        outpath = os.path.join(heat_dir, f"{tag}_HEAT_median_{metric}.png")
        plt.savefig(outpath, dpi=DPI)
        plt.close()
        return True

    for m in ["energy_jump_ratio", "rise_time_sec", "centroid_slope_hz_per_s"]:
        cap_upper = (m != "centroid_slope_hz_per_s")
        if m in data.columns:
            if heatmap_median(m, cap_upper=cap_upper):
                count += 1

    return count

# =========================
# ====== GENERATE ALL =====
# =========================
total = 0
total += generate_all_for_dataset("MAIN", df_main, main_dir)

if len(df_sens) >= 10:
    total += generate_all_for_dataset("SENS", df_sens, supp_dir)

print("\nDONE ✅")
print(f"Total figures saved (approx): {total}")
print("Output folder:", OUT)
print("\nTip: MAIN figures are in MAIN/* and Supplementary in SUPPLEMENTARY/*")


MAIN rows: 80 | SENS rows: 183
Leaks: ['NL', 'OL', 'GL', 'LC', 'CC', 'NOISE']
Sensors: ['Accelerometer', 'Dynamic Pressure Sensor', 'Hydrophone']
Topologies: ['BR', 'LO', 'BACKGROUND']
MAIN flows: ['TRANSIENT']

DONE ✅
Total figures saved (approx): 112
Output folder: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_50PLUS_FIGURES

Tip: MAIN figures are in MAIN/* and Supplementary in SUPPLEMENTARY/*


In [14]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import signal

# ==========================================================
# ===================== USER CONFIG =========================
# ==========================================================
DATA_ROOTS = [
    r"E:\Upwork Project\AI_Leak_Detection_Project\data\raw\Accelerometer",
    r"E:\Upwork Project\AI_Leak_Detection_Project\data\raw\Dynamic Pressure Sensor",
    r"E:\Upwork Project\AI_Leak_Detection_Project\data\raw\Hydrophone",
]

SUMMARY_CSV = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\transient_energy_summary.csv"

OUT_ROOT = r"E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_PAPER_FACTORY"
os.makedirs(OUT_ROOT, exist_ok=True)

# Sampling assumptions (adjust if you know exact)
FS_HYDROPHONE = 44100
FS_CSV_DEFAULT = 10240
DURATION_SEC = 30.0

# STFT settings (robust)
STFT_NPERSEG = 2048
STFT_NOVERLAP = 1536

# Frequency bands (Hz)
BANDS = {
    "low": (0.0, 200.0),
    "mid": (200.0, 1000.0),
    "high": (1000.0, 5000.0),
}

# Plot style
DPI = 350
CAP_Q = 0.98
MIN_N = 5

LEAK_ORDER = ["NL", "OL", "GL", "LC", "CC"]   # dataset acronyms
SENSOR_ORDER = ["Accelerometer", "Dynamic Pressure Sensor", "Hydrophone"]
TOPO_ORDER = ["BR", "LO"]

# ==========================================================
# ===================== MSSP STYLE ==========================
# ==========================================================
rcParams["font.family"] = "serif"
rcParams["font.size"] = 13
rcParams["axes.labelweight"] = "bold"
rcParams["axes.titleweight"] = "bold"
rcParams["axes.linewidth"] = 1.8
rcParams["xtick.major.width"] = 1.4
rcParams["ytick.major.width"] = 1.4
rcParams["legend.frameon"] = False
rcParams["figure.facecolor"] = "white"

def safe_mkdir(p):
    os.makedirs(p, exist_ok=True)
    return p

def tight_save(path):
    plt.tight_layout()
    plt.savefig(path, dpi=DPI)
    plt.close()

def cap(vals, upper=True, q=CAP_Q):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return vals, None
    c = np.nanquantile(vals, q)
    if upper:
        return np.clip(vals, 0, c), c
    return np.clip(vals, -c, c), c

# ==========================================================
# ===================== LOAD SUMMARY ========================
# ==========================================================
df = pd.read_csv(SUMMARY_CSV)
if "error" in df.columns:
    df = df[df["error"].isna()].copy()

metrics = [
    "energy_jump_ratio",
    "rise_time_sec",
    "centroid_slope_hz_per_s",
    "centroid_mean_hz",
    "band_jump_low","band_jump_mid","band_jump_high",
]

for m in metrics:
    if m in df.columns:
        df[m] = pd.to_numeric(df[m], errors="coerce")

# MAIN: labeled transient (reviewer-proof)
df_main = df[df["flow"].astype(str).str.contains("TRANSIENT", case=False, na=False)].copy()
# SENS: detected transient (appendix)
df_sens = df[df["transient_detected"] == True].copy()

if len(df_main) < 10:
    print("[WARN] Labeled transient subset too small. Using detected transient as MAIN.")
    df_main, df_sens = df_sens, df_main

# Orders only for present categories
leaks_present = [l for l in LEAK_ORDER if l in set(df["leak"].astype(str))]
leaks_present += [l for l in sorted(df["leak"].dropna().unique()) if l not in leaks_present]

sensors_present = [s for s in SENSOR_ORDER if s in set(df["sensor_group"].astype(str))]
sensors_present += [s for s in sorted(df["sensor_group"].dropna().unique()) if s not in sensors_present]

topos_present = [t for t in TOPO_ORDER if t in set(df["topology"].astype(str))]
topos_present += [t for t in sorted(df["topology"].dropna().unique()) if t not in topos_present]

print(f"MAIN rows: {len(df_main)} | SENS rows: {len(df_sens)}")
print("Leaks:", leaks_present)
print("Sensors:", sensors_present)
print("Topologies:", topos_present)

# ==========================================================
# ========= TABLES: PAPER + SUPPLEMENTARY ===================
# ==========================================================
TAB_DIR = safe_mkdir(os.path.join(OUT_ROOT, "TABLES"))

def table_stats(dfx, group_cols, metrics_list, out_csv):
    agg = {}
    for m in metrics_list:
        if m in dfx.columns:
            agg[m] = ["count","median","mean","std"]
    t = dfx.groupby(group_cols, dropna=False).agg(agg)
    t.columns = ["_".join(c) for c in t.columns.to_flat_index()]
    t = t.reset_index()
    t.to_csv(out_csv, index=False)
    return t

table_stats(df_main, ["sensor_group","leak"], metrics, os.path.join(TAB_DIR,"TABLE_MAIN_sensor_leak.csv"))
table_stats(df_main, ["sensor_group","topology","leak"], metrics, os.path.join(TAB_DIR,"TABLE_MAIN_sensor_topology_leak.csv"))
table_stats(df_main, ["sensor_group","topology","leak","flow"], metrics, os.path.join(TAB_DIR,"TABLE_MAIN_sensor_topology_leak_flow.csv"))

if len(df_sens) >= 10:
    table_stats(df_sens, ["sensor_group","leak"], metrics, os.path.join(TAB_DIR,"TABLE_SENS_sensor_leak.csv"))

# ==========================================================
# ============ FIGURES FROM SUMMARY (FAST) ===================
# ==========================================================
FIG_MAIN = safe_mkdir(os.path.join(OUT_ROOT, "FIGURES_MAIN"))
FIG_SUPP = safe_mkdir(os.path.join(OUT_ROOT, "FIGURES_SUPP"))
FIG_HEAT = safe_mkdir(os.path.join(OUT_ROOT, "FIGURES_HEATMAPS"))
FIG_DIST = safe_mkdir(os.path.join(OUT_ROOT, "FIGURES_DISTRIBUTIONS"))

def boxplot_sensor_leak(dfx, metric, outpath, title):
    d = dfx.dropna(subset=["sensor_group","leak",metric]).copy()
    d = d[np.isfinite(d[metric].astype(float))]
    if len(d) < 10:
        return
    upper = (metric not in ["centroid_slope_hz_per_s"])
    allv = d[metric].astype(float).values
    capv = np.nanquantile(allv, CAP_Q)
    d["_m"] = np.clip(d[metric].astype(float), 0 if upper else -capv, capv)

    positions, data, xticks, xlabels = [], [], [], []
    pos = 0
    for s in sensors_present:
        ds = d[d["sensor_group"] == s]
        for l in leaks_present:
            v = ds[ds["leak"] == l]["_m"].values
            if len(v) >= MIN_N:
                data.append(v); positions.append(pos)
                xticks.append(pos)
                s_short = s.replace("Dynamic Pressure Sensor","Pressure").replace("Accelerometer","Accel").replace("Hydrophone","Hydro")
                xlabels.append(f"{s_short}\n{l}")
            pos += 1
        pos += 1

    if len(data) == 0:
        return
    plt.figure(figsize=(14,5))
    plt.boxplot(data, positions=positions, widths=0.6, showfliers=False)
    plt.xticks(xticks, xlabels)
    plt.ylabel(metric.replace("_"," ").title())
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.25)
    tight_save(outpath)

def violin_sensor_leak(dfx, metric, outpath, title):
    d = dfx.dropna(subset=["sensor_group","leak",metric]).copy()
    d = d[np.isfinite(d[metric].astype(float))]
    if len(d) < 10:
        return
    upper = (metric not in ["centroid_slope_hz_per_s"])
    allv = d[metric].astype(float).values
    capv = np.nanquantile(allv, CAP_Q)
    d["_m"] = np.clip(d[metric].astype(float), 0 if upper else -capv, capv)

    data, positions, xticks, xlabels = [], [], [], []
    pos = 0
    for s in sensors_present:
        ds = d[d["sensor_group"] == s]
        for l in leaks_present:
            v = ds[ds["leak"] == l]["_m"].values
            if len(v) >= MIN_N:
                data.append(v); positions.append(pos)
                xticks.append(pos)
                s_short = s.replace("Dynamic Pressure Sensor","Pressure").replace("Accelerometer","Accel").replace("Hydrophone","Hydro")
                xlabels.append(f"{s_short}\n{l}")
            pos += 1
        pos += 1
    if len(data) == 0:
        return

    plt.figure(figsize=(14,5))
    vp = plt.violinplot(data, positions=positions, widths=0.8, showmeans=False, showmedians=True, showextrema=False)
    for b in vp["bodies"]:
        b.set_edgecolor("black")
        b.set_linewidth(0.8)
        b.set_alpha(0.85)
    plt.xticks(xticks, xlabels)
    plt.ylabel(metric.replace("_"," ").title())
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.25)
    tight_save(outpath)

def heatmap_median(dfx, metric, outpath, title):
    d = dfx.dropna(subset=["sensor_group","leak",metric]).copy()
    d = d[np.isfinite(d[metric].astype(float))]
    if len(d) < 10:
        return
    piv = d.pivot_table(index="sensor_group", columns="leak", values=metric, aggfunc="median")
    piv = piv.reindex(index=sensors_present, columns=leaks_present)
    arr = piv.to_numpy(dtype=float)
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return

    upper = (metric not in ["centroid_slope_hz_per_s"])
    capv = np.nanquantile(finite, CAP_Q)
    arr = np.clip(arr, 0 if upper else -capv, capv)

    plt.figure(figsize=(7.8,4.8))
    im = plt.imshow(arr, aspect="auto")
    plt.xticks(range(len(leaks_present)), leaks_present)
    s_short = [s.replace("Dynamic Pressure Sensor","Pressure").replace("Accelerometer","Accel").replace("Hydrophone","Hydro") for s in sensors_present]
    plt.yticks(range(len(sensors_present)), s_short)
    plt.title(title)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    tight_save(outpath)

def band_profile(dfx, outpath, title):
    needed = ["band_jump_low","band_jump_mid","band_jump_high"]
    d = dfx.dropna(subset=["sensor_group","leak"] + needed).copy()
    for c in needed:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=needed)
    if len(d) < 10:
        return
    # cap each band
    for c in needed:
        capv = np.nanquantile(d[c].astype(float), CAP_Q)
        d[c] = np.clip(d[c].astype(float), 0, capv)
    g = d.groupby(["sensor_group","leak"])[needed].median().reset_index()

    plt.figure(figsize=(15,5))
    width = 0.22
    x = 0
    xticks, labels = [], []

    # legend
    plt.bar(-10, 0, width=width, label="Low")
    plt.bar(-10, 0, width=width, label="Mid")
    plt.bar(-10, 0, width=width, label="High")

    for s in sensors_present:
        gs = g[g["sensor_group"] == s]
        for l in leaks_present:
            row = gs[gs["leak"] == l]
            if len(row) == 0:
                x += 1
                continue
            low, mid, high = row[needed].values[0]
            plt.bar(x - width, low, width=width)
            plt.bar(x, mid, width=width)
            plt.bar(x + width, high, width=width)
            xticks.append(x)
            s_short = s.replace("Dynamic Pressure Sensor","Pressure").replace("Accelerometer","Accel").replace("Hydrophone","Hydro")
            labels.append(f"{s_short}\n{l}")
            x += 1
        x += 1

    plt.xticks(xticks, labels)
    plt.ylabel("Median band-energy jump ratio (capped)")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.25)
    plt.legend(ncol=3, loc="upper right")
    tight_save(outpath)

# MAIN figures (box + violin + heatmaps + band profiles)
for m in ["energy_jump_ratio","rise_time_sec","centroid_slope_hz_per_s","band_jump_low","band_jump_mid","band_jump_high"]:
    if m in df_main.columns:
        boxplot_sensor_leak(df_main, m, os.path.join(FIG_MAIN, f"MAIN_BOX_{m}.png"),
                            f"MAIN (Labeled transient): {m.replace('_',' ').title()} by sensor and leak")
        violin_sensor_leak(df_main, m, os.path.join(FIG_MAIN, f"MAIN_VIOLIN_{m}.png"),
                           f"MAIN (Labeled transient): {m.replace('_',' ').title()} by sensor and leak")
        heatmap_median(df_main, m, os.path.join(FIG_HEAT, f"MAIN_HEAT_MEDIAN_{m}.png"),
                       f"MAIN: Median {m.replace('_',' ').title()} (Sensor × Leak)")

band_profile(df_main, os.path.join(FIG_MAIN, "MAIN_BAND_PROFILE.png"),
             "MAIN: Median band-energy propagation profile by sensor and leak")

# SUPP sensitivity versions
if len(df_sens) >= 10:
    for m in ["energy_jump_ratio","rise_time_sec","centroid_slope_hz_per_s","band_jump_low","band_jump_mid","band_jump_high"]:
        if m in df_sens.columns:
            boxplot_sensor_leak(df_sens, m, os.path.join(FIG_SUPP, f"SENS_BOX_{m}.png"),
                                f"SENS (Detected transient): {m.replace('_',' ').title()} by sensor and leak")
            heatmap_median(df_sens, m, os.path.join(FIG_HEAT, f"SENS_HEAT_MEDIAN_{m}.png"),
                           f"SENS: Median {m.replace('_',' ').title()} (Sensor × Leak)")
    band_profile(df_sens, os.path.join(FIG_SUPP, "SENS_BAND_PROFILE.png"),
                 "SENS: Median band-energy propagation profile by sensor and leak")

# ==========================================================
# ====== RAW SIGNAL FIGURES (Waveform/PSD/STFT/Energy) ======
# ==========================================================
RAW_DIR = safe_mkdir(os.path.join(OUT_ROOT, "FIGURES_RAW_SIGNALS"))
PANEL_DIR = safe_mkdir(os.path.join(OUT_ROOT, "FIGURES_PANELS"))

def load_csv_signal(path):
    dfc = pd.read_csv(path)
    if dfc.shape[1] == 1:
        x = pd.to_numeric(dfc.iloc[:,0], errors="coerce").to_numpy()
        fs = FS_CSV_DEFAULT
        x = x[np.isfinite(x)]
        return x.astype(float), fs
    # choose last numeric column
    numeric = []
    for c in dfc.columns:
        s = pd.to_numeric(dfc[c], errors="coerce")
        if s.notna().mean() > 0.8:
            numeric.append(c)
    if not numeric:
        x = dfc.to_numpy().reshape(-1)
        x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy()
        x = x[np.isfinite(x)]
        return x.astype(float), FS_CSV_DEFAULT
    x = pd.to_numeric(dfc[numeric[-1]], errors="coerce").to_numpy()
    x = x[np.isfinite(x)]
    return x.astype(float), FS_CSV_DEFAULT

def load_raw_hydro(path):
    x = np.fromfile(path, dtype=np.int16).astype(np.float32)
    if np.max(np.abs(x)) > 0:
        x = x / (np.max(np.abs(x)) + 1e-12)
    return x.astype(float), FS_HYDROPHONE

def trim30(x, fs):
    n = int(round(fs*DURATION_SEC))
    if len(x) >= n:
        return x[:n]
    y = np.zeros(n, dtype=float)
    y[:len(x)] = x
    return y

def stft_pack(x, fs):
    nperseg = min(STFT_NPERSEG, len(x))
    noverlap = min(STFT_NOVERLAP, nperseg-1) if nperseg>1 else 0
    f,t,Z = signal.stft(x, fs=fs, window="hann", nperseg=nperseg, noverlap=noverlap, padded=False, boundary=None)
    P = (np.abs(Z)**2)
    E = np.sum(P, axis=0) + 1e-12
    centroid = np.sum(f[:,None]*P, axis=0) / (np.sum(P, axis=0)+1e-12)
    bands = {}
    for bn,(lo,hi) in BANDS.items():
        idx = np.where((f>=lo) & (f<hi))[0]
        bands[bn] = np.sum(P[idx,:], axis=0) if len(idx) else np.zeros_like(E)
    return f,t,P,E,centroid,bands

def plot_wave_psd_stft_energy(x, fs, title, outpath, t0=None):
    x = x - np.mean(x)
    # light highpass
    try:
        sos = signal.butter(4, 1.0, btype="highpass", fs=fs, output="sos")
        x = signal.sosfiltfilt(sos, x)
    except Exception:
        pass

    f_stft,t_stft,P,E,centroid,bands = stft_pack(x, fs)

    tt = np.arange(len(x))/fs
    f_psd, Pxx = signal.welch(x, fs=fs, nperseg=min(8192, len(x)))

    plt.figure(figsize=(13,10))
    plt.suptitle(title, fontsize=14, fontweight="bold")

    ax1 = plt.subplot(4,1,1)
    ax1.plot(tt, x, linewidth=1.1)
    if t0 is not None: ax1.axvline(t0, linestyle="--", linewidth=1.2)
    ax1.set_ylabel("Amplitude")
    ax1.grid(True, alpha=0.25)

    ax2 = plt.subplot(4,1,2)
    ax2.semilogy(f_psd, Pxx+1e-18, linewidth=1.2)
    ax2.set_xlim([0, min(5000, fs/2)])
    ax2.set_ylabel("PSD")
    ax2.grid(True, alpha=0.25)

    ax3 = plt.subplot(4,1,3)
    Pdb = 10*np.log10(P+1e-12)
    ax3.pcolormesh(t_stft, f_stft, Pdb, shading="gouraud")
    if t0 is not None: ax3.axvline(t0, linestyle="--", linewidth=1.2)
    ax3.set_ylim([0, min(5000, fs/2)])
    ax3.set_ylabel("Hz")
    ax3.set_title("STFT Spectrogram (dB)")

    ax4 = plt.subplot(4,1,4)
    ax4.plot(t_stft, np.log(E), label="log Energy", linewidth=1.3)
    ax4.plot(t_stft, bands["low"]/np.max(bands["low"]+1e-12), label="Low band (norm)", linewidth=1.0)
    ax4.plot(t_stft, bands["mid"]/np.max(bands["mid"]+1e-12), label="Mid band (norm)", linewidth=1.0)
    ax4.plot(t_stft, bands["high"]/np.max(bands["high"]+1e-12), label="High band (norm)", linewidth=1.0)
    if t0 is not None: ax4.axvline(t0, linestyle="--", linewidth=1.2)
    ax4.set_xlabel("Time (s)")
    ax4.set_ylabel("Energy / Norm band")
    ax4.legend(ncol=4, loc="upper right")
    ax4.grid(True, alpha=0.25)

    tight_save(outpath)

# ---- pick representative examples automatically (best-of-the-best) ----
# Strategy:
# - For each sensor and each leak type, pick the file with the largest energy_jump_ratio in MAIN.
# - Also pick a "no-leak" reference per sensor.
EX_DIR = safe_mkdir(os.path.join(OUT_ROOT, "EXAMPLES_SELECTED"))
df_ex = df_main.dropna(subset=["path","sensor_group","leak","energy_jump_ratio"]).copy()

# make sure files exist
df_ex = df_ex[df_ex["path"].astype(str).apply(lambda p: os.path.exists(p))]

selected = []
for s in sensors_present:
    for l in leaks_present:
        dl = df_ex[(df_ex["sensor_group"]==s) & (df_ex["leak"]==l)]
        if len(dl) == 0:
            continue
        # pick max jump ratio as "strong example"
        row = dl.sort_values("energy_jump_ratio", ascending=False).iloc[0]
        selected.append(row)

# also pick median examples (more realistic) for each sensor-leak
for s in sensors_present:
    for l in leaks_present:
        dl = df_ex[(df_ex["sensor_group"]==s) & (df_ex["leak"]==l)]
        if len(dl) < 3:
            continue
        row = dl.sort_values("energy_jump_ratio", ascending=True).iloc[len(dl)//2]
        selected.append(row)

sel_df = pd.DataFrame(selected).drop_duplicates(subset=["path"])
sel_df.to_csv(os.path.join(EX_DIR, "selected_examples.csv"), index=False)
print(f"Selected example signals: {len(sel_df)}")

# ---- generate raw-signal figures (waveform + PSD + STFT + energy bands) ----
max_raw_figs = 60  # keep it bounded; still plenty for main+supp
count = 0
for _,r in sel_df.iterrows():
    if count >= max_raw_figs:
        break
    path = str(r["path"])
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv":
            x, fs = load_csv_signal(path)
        elif ext == ".raw":
            x, fs = load_raw_hydro(path)
        else:
            continue
        x = trim30(x, fs)
        title = f"{r['sensor_group']} | {r['topology']} {r['leak']} | Flow={r['flow']} | {os.path.basename(path)}"
        outp = os.path.join(RAW_DIR, f"RAW_{count:03d}_{r['sensor_group'].replace(' ','_')}_{r['leak']}_{r['topology']}.png")
        plot_wave_psd_stft_energy(x, fs, title, outp, t0=float(r["t0"]) if "t0" in r and np.isfinite(r["t0"]) else None)
        count += 1
    except Exception as e:
        continue

print(f"Raw-signal composite figures saved: {count}")

# ==========================================================
# ========== MAIN MULTI-PANEL “PAPER” PANELS ================
# ==========================================================
# Create 3x2 panel for each leak: (Accel/Pressure/Hydro) x (No-leak vs leak)
# Uses best examples found above.
PANELS = safe_mkdir(os.path.join(PANEL_DIR, "MAIN_3x2_PANELS"))

def pick_best(sensor, leak):
    d = df_ex[(df_ex["sensor_group"]==sensor) & (df_ex["leak"]==leak)]
    if len(d)==0: return None
    d = d[d["path"].astype(str).apply(os.path.exists)]
    if len(d)==0: return None
    return d.sort_values("energy_jump_ratio", ascending=False).iloc[0]

def load_signal(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        x, fs = load_csv_signal(path)
    else:
        x, fs = load_raw_hydro(path)
    x = trim30(x, fs)
    x = x - np.mean(x)
    return x, fs

def panel_3x2(leak_code):
    # left col = NL, right col = leak_code
    rows = []
    for s in sensors_present:
        rNL = pick_best(s, "NL")
        rLK = pick_best(s, leak_code)
        if rNL is None or rLK is None:
            return
        rows.append((s, rNL, rLK))

    plt.figure(figsize=(14, 10))
    for i,(s,rNL,rLK) in enumerate(rows):
        for j,(r,lab) in enumerate([(rNL,"No-leak"),(rLK,leak_code)]):
            path = str(r["path"])
            x,fs = load_signal(path)
            f_stft,t_stft,P,E,centroid,bands = stft_pack(x, fs)
            Pdb = 10*np.log10(P+1e-12)
            ax = plt.subplot(3,2,i*2+j+1)
            ax.pcolormesh(t_stft, f_stft, Pdb, shading="gouraud")
            ax.set_ylim([0, min(5000, fs/2)])
            if "t0" in r and np.isfinite(r["t0"]):
                ax.axvline(float(r["t0"]), linestyle="--", linewidth=1.2, color="white")
            ax.set_title(f"{s.replace('Dynamic Pressure Sensor','Pressure')} | {lab}", fontweight="bold")
            if j==0: ax.set_ylabel("Hz")
            if i==2: ax.set_xlabel("Time (s)")

    plt.suptitle(f"STFT Spectrogram Panels (No-leak vs {leak_code})", fontweight="bold", fontsize=16)
    tight_save(os.path.join(PANELS, f"PANEL_STFT_3x2_NL_vs_{leak_code}.png"))

for lk in [l for l in leaks_present if l!="NL"]:
    panel_3x2(lk)

print("DONE ✅")
print("All outputs saved under:", OUT_ROOT)
print("Main figures:", FIG_MAIN)
print("Supplementary figures:", FIG_SUPP)
print("Heatmaps:", FIG_HEAT)
print("Raw signal composites:", RAW_DIR)
print("Multi-panels:", PANEL_DIR)
print("Tables:", TAB_DIR)


MAIN rows: 80 | SENS rows: 183
Leaks: ['NL', 'OL', 'GL', 'LC', 'CC', 'NOISE']
Sensors: ['Accelerometer', 'Dynamic Pressure Sensor', 'Hydrophone']
Topologies: ['BR', 'LO', 'BACKGROUND']
Selected example signals: 30
Raw-signal composite figures saved: 30
DONE ✅
All outputs saved under: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_PAPER_FACTORY
Main figures: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_PAPER_FACTORY\FIGURES_MAIN
Supplementary figures: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_PAPER_FACTORY\FIGURES_SUPP
Heatmaps: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_PAPER_FACTORY\FIGURES_HEATMAPS
Raw signal composites: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_PAPER_FACTORY\FIGURES_RAW_SIGNALS
Multi-panels: E:\Upwork Project\AI_Leak_Detection_Project\outputs_transient_energy\MSSP_PAPER_FACTORY\FIGURES_PANELS
Tables: E:\Upwork Project